# Factor risk models from returns alone

This notebook builds a factor risk model from a return panel in two ways:

- **Statistical (PCA, nonlinear shrinkage).** The return covariance gives both the factors
  and the loadings. The method needs no outside data and gives no interpretation.
- **Fama-French time series.** Published factor return series (Kenneth French, free of
  charge) are the factors. A regression of the excess returns of each name on these
  factors gives its loadings. The loadings come from returns alone, and the factors have
  names: market, size, value, profitability, investment and momentum.

The notebook compares the Fama-French covariance with the sample covariance, nonlinear
shrinkage (QIS) and the PCA statistical model from `sdp.risk`. The Fama-French factors are
realised return series, so they are point in time. They lag the live data by about two
months, so the most recent sessions are not in the sample.

In [ ]:
import io
import urllib.request
import zipfile

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl

from sdp import dal
from sdp.risk import eligible, ff_cov, gmv, pca_cov, qis, sample_cov

wh = dal.warehouse()

## The return panel

The panel holds liquid common stock (`in_universe`) and uses the same load as
`covariance.ipynb`. It loads each name that is in the universe on one or more dates of the
window. A name has NaN on each session with no price.

The selection of names is point in time. At each rebalance date, a name is eligible when
it is in the universe on that date and has a return on each session of the estimation
window. The estimate uses the `N_NAMES` eligible names with the largest trailing dollar
volume on that date. A held name with no return on a session of the hold period
contributes a zero return on that session. This rule is an approximation. For a name that
leaves the tape, the weight earns zero from that session, and the backtest does not see a
delisting return. For a name with a gap in its prices, the backtest does not see the move
across the gap. The rule does not carry a price over a gap, because the panel keys on the
ticker and a ticker can pass to a new security at a merger close.

In [ ]:
N_NAMES = 1000     # names in each estimate
TOTAL = 780        # trailing sessions of prices

panel = wh.sql(f"""
    with win as (
        select distinct date from main_staging.stg_universe
        order by date desc limit {TOTAL}
    ), names as (
        select distinct ticker from main_staging.stg_universe
        where in_universe and date >= (select min(date) from win)
    )
    select p.date, p.ticker, p.adj_close_total,
           coalesce(u.in_universe, false) as in_universe, u.adv
    from main_staging.stg_prices_adjusted p
    join names using (ticker)
    left join main_staging.stg_universe u using (ticker, date)
    where p.date >= (select min(date) from win)
""").pl()


def wide(col):
    """Pivot one panel column to a sessions x names frame, with the names in sorted order."""
    return panel.pivot(on="ticker", index="date", values=col, sort_columns=True).sort("date")


px_wide = wide("adj_close_total")
rdates = px_wide["date"].to_numpy()[1:]                           # date of each return row
R_all = np.diff(np.log(px_wide.drop("date").to_numpy()), axis=0)  # NaN where no price
U_all = wide("in_universe").drop("date").fill_null(False).to_numpy()[1:]
A_all = wide("adv").drop("date").fill_null(0.0).to_numpy()[1:]

print(f"panel {R_all.shape}, {rdates[0]} -> {rdates[-1]}")

## Fama-French factors

The Kenneth French Data Library gives the five factors (`Mkt-RF`, `SMB`, `HML`, `RMW`,
`CMA`) and momentum, daily and in percent, free of charge. The loader keeps only the daily
rows (an 8-digit date) and ignores the annual section at the end of the file.

The structure sections use one matrix: the names that are eligible on the last session
with factors, with all the sessions with factors as the estimation window.

In [ ]:
def french(url, valcols):
    """Download one Kenneth French daily factor file into a polars frame."""
    raw = urllib.request.urlopen(
        urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=30).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    txt = z.read(z.namelist()[0]).decode("latin-1")
    rows = []
    for ln in txt.splitlines():
        p = [x.strip() for x in ln.split(",")]
        if len(p) == len(valcols) + 1 and p[0].isdigit() and len(p[0]) == 8:
            rows.append([p[0]] + [float(x) / 100 for x in p[1:]])   # percent -> fraction
    return (pl.DataFrame(rows, schema=["d"] + valcols, orient="row")
            .with_columns(pl.col("d").str.strptime(pl.Date, "%Y%m%d"))
            .rename({"d": "date"}))


BASE = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
ff5 = french(BASE + "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip",
             ["Mkt_RF", "SMB", "HML", "RMW", "CMA", "RF"])
mom = french(BASE + "F-F_Momentum_Factor_daily_CSV.zip", ["Mom"])
fac = ff5.join(mom, on="date", how="inner")
FCOLS = ["Mkt_RF", "SMB", "HML", "RMW", "CMA", "Mom"]

# Align to the panel dates, in date order. A session with no factor row is not in the sample.
aligned = (pl.DataFrame({"date": rdates}).with_row_index("i")
           .join(fac, on="date", how="inner").sort("i"))
idx = aligned["i"].to_numpy()
R_ff = R_all[idx]                              # returns on the sessions that have factors
U_ff, A_ff = U_all[idx], A_all[idx]
F = aligned.select(FCOLS).to_numpy()           # T x K factor returns
rf = aligned["RF"].to_numpy()                  # daily risk-free rate
T = len(idx)
X = R_ff[:, eligible(R_ff, U_ff, A_ff, T, T, N_NAMES)]  # structure matrix, T x N
N = X.shape[1]
print(f"factors {fac['date'].min()} -> {fac['date'].max()}")
print(f"aligned T={T} N={N} q=N/T={N / T:.2f} "
      f"({len(rdates) - T} sessions with no factor row are not in the sample)")

## The time-series factor model

The model regresses the excess return of each name on the factors over the window:

$r_{i,t} - r^f_t = \alpha_i + \sum_k \beta_{i,k}\, f_{k,t} + \varepsilon_{i,t}$

The betas $B$ (N x K) are the loadings, $\Omega$ is the covariance of the factor returns,
and the idiosyncratic variances are the residual variances. The risk model is
$\Sigma = B\,\Omega\,B^\top + \operatorname{diag}(\sigma^2_\varepsilon)$.

The model assumes that the residuals are not correlated across names (a diagonal). This is
the main limitation of the model: it drops each common structure that the six factors do
not capture, for example industry.

In [ ]:
Sig_ff, B = ff_cov(X, F, rf)
print("average loading per factor (cross-sectional mean beta):")
for k, name in enumerate(FCOLS):
    print(f"  {name:7s} {B[:, k].mean():+.3f}")
print(f"\nmarket-beta average {B[:, 0].mean():.2f} (expected near 1)")

fig = px.bar(x=FCOLS, y=np.abs(B).mean(0),
             title="Average absolute factor loading across the universe")
fig.update_layout(xaxis_title="", yaxis_title="mean |beta|", height=380)
fig.show()

## Structure: conditioning and spectra

The comparison uses three estimators from `sdp.risk`: the sample covariance, nonlinear
shrinkage (QIS) and the PCA statistical factor model. When N is near T or larger, the
sample matrix is singular or near singular. The Fama-French model is six factors plus a
diagonal, so its spectrum has at most six large eigenvalues above a floor that the
idiosyncratic variances set.

In [ ]:
rows = []
for name, S in [("sample", sample_cov(X)), ("fama_french", Sig_ff),
                ("qis", qis(X)), ("pca", pca_cov(X))]:
    ev = np.linalg.eigvalsh(S)
    rows.append({"method": name, "condition_number": float(np.linalg.cond(S)),
                 "posdef": bool(ev.min() > 0), "min_eig": float(ev.min())})
pl.DataFrame(rows).sort("condition_number")

In [ ]:
fig = go.Figure()
for name, S in [("sample", sample_cov(X)), ("fama_french", Sig_ff),
                ("qis", qis(X)), ("pca", pca_cov(X))]:
    ev = np.sort(np.linalg.eigvalsh(S))[::-1]
    fig.add_trace(go.Scatter(y=np.clip(ev, 1e-12, None), mode="lines", name=name))
fig.update_yaxes(type="log", title="eigenvalue (log)")
fig.update_layout(title="Covariance eigenvalue spectra", xaxis_title="rank", height=440)
fig.show()

## Economic test: out-of-sample minimum-variance risk

At each rebalance date, the backtest selects the eligible names, estimates each covariance
on the trailing window, forms the global minimum-variance portfolio and holds it to the
next rebalance. The score is the realised volatility of the out-of-sample returns. With
`N_NAMES = 1000`, `EST_WIN = 252` puts q = N/win well above 1, which is the regime that
separates the methods.

In [ ]:
def backtest(kind, win, step=21):
    """Return the annualised out-of-sample GMV volatility in percent and the mean count of
    names, over the rebalances."""
    oos, size = [], []
    for d in range(win, T - 1, step):
        cols = eligible(R_ff, U_ff, A_ff, d, win, N_NAMES)
        Rw = R_ff[d - win:d, cols]
        if kind == "fama_french":
            S = ff_cov(Rw, F[d - win:d], rf[d - win:d])[0]
        else:
            S = {"sample": sample_cov, "qis": qis, "pca": pca_cov}[kind](Rw)
        size.append(cols.size)
        oos.append(np.nan_to_num(R_ff[d:d + step, cols]) @ gmv(S))   # no return: zero
    return float(np.concatenate(oos).std() * np.sqrt(252) * 100), float(np.mean(size))


EST_WIN = 252
rows = []
for kind in ("fama_french", "qis", "pca", "sample"):
    vol, n = backtest(kind, EST_WIN)
    rows.append({"method": kind, "oos_vol_pct": round(vol, 2)})
res = pl.DataFrame(rows).sort("oos_vol_pct")
print(f"EST_WIN={EST_WIN}  mean N={n:.0f}  q=N/win={n / EST_WIN:.1f}")
res

In [ ]:
plot = res.filter(pl.col("oos_vol_pct") < 100)          # keep a blow-up off the scale
omitted = sorted(set(res["method"]) - set(plot["method"]))
title = f"Out-of-sample GMV volatility (EST_WIN={EST_WIN})"
if omitted:
    title += f" ({', '.join(omitted)} omitted, off scale)"
fig = px.bar(plot.to_pandas(), x="method", y="oos_vol_pct", title=title)
fig.update_layout(yaxis_title="annualised vol %", xaxis_title="", height=420)
fig.show()

## Takeaways

- **Fama-French is the interpretable risk model.** It has six named factors and gets its
  loadings from returns alone, with no characteristic data. It is positive definite and
  well conditioned, also when N>T. Its factor covariance and loadings give a risk
  attribution that a statistical model cannot give.
- **It is coarser than the statistical models on pure variance.** Six factors and a
  diagonal residual do not capture the common structure that QIS and PCA find. Thus its
  out-of-sample GMV volatility is higher than theirs, but far below that of the sample
  matrix. The diagonal residual is the likely cause. A residual-shrinkage step
  (POET-style) may close part of the gap.
- **Use it for attribution and hedging, and use the statistical models for the
  optimiser.** The next step is a fundamental (Barra-style) model with characteristic
  loadings (size, value, industry). It needs the ticker-details and fundamentals data,
  which the platform does not ingest yet.

The Fama-French factors lag the live data by about two months. Thus a live risk model
uses them for the loadings and a statistical model for the most recent sessions.